# Qualidade dos Dados — ANP

Neste notebook analiso a qualidade dos dados de preços de combustíveis disponibilizados pela ANP.

O objetivo é investigar e validar identificadores importantes, como CNPJ e CEP, preservando os valores recebidos da fonte e aplicando regras de normalização somente quando a correção puder ser realizada com segurança.

As regras confirmadas durante a análise são implementadas em módulos reutilizáveis dentro de `src/qualidade`, permitindo que posteriormente sejam utilizadas pelo pipeline de dados.

In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

import pandas as pd


In [ ]:
# Identifico onde o notebook está sendo executado
caminho_atual = Path.cwd()

# Ajusto a raiz caso o kernel tenha iniciado dentro da pasta notebooks
if caminho_atual.name == "notebooks":
    caminho_projeto = caminho_atual.parent
else:
    caminho_projeto = caminho_atual

print("Diretório atual:", caminho_atual)
print("Raiz do projeto:", caminho_projeto)


In [ ]:
# Adiciono a raiz do projeto ao caminho de importação
# para reutilizar os módulos desenvolvidos em src
if str(caminho_projeto) not in sys.path:
    sys.path.insert(0, str(caminho_projeto))


In [ ]:
# Importo as regras reutilizáveis de Data Quality
from src.qualidade.validar_cnpj import (
    limpar_cnpj,
    normalizar_cnpj,
    validar_cnpj,
)

from src.qualidade.validar_cep import (
    classificar_cep,
    normalizar_cep,
    validar_formato_cep,
)

from src.qualidade.classificar_registro import (
    avaliar_qualidade_registro,
)

from src.qualidade.preparar_quarentena import (
    preparar_quarentena,
)


In [ ]:
# Defino a amostra da ANP utilizada nesta investigação
caminho_arquivo = (
    caminho_projeto
    / "dados"
    / "amostras"
    / "revendas_lpc_2026-09-06_2026-09-12.xlsx"
)

nome_planilha = "POSTOS REVENDEDORES"
linha_cabecalho = 9

print("Arquivo encontrado:", caminho_arquivo.exists())


In [ ]:
# Preservo CNPJ e CEP como texto porque são identificadores,
# e não valores numéricos utilizados em cálculos
tipos_identificadores = {
    "CNPJ": "string",
    "CEP": "string",
}

dados = pd.read_excel(
    caminho_arquivo,
    sheet_name=nome_planilha,
    header=linha_cabecalho,
    dtype=tipos_identificadores,
)

print(f"Quantidade de registros: {len(dados):,}")
print(f"Quantidade de colunas: {len(dados.columns)}")

In [ ]:
# Registro a posição da linha original para permitir rastreamento até o arquivo recebido da ANP
dados["linha_origem"] = dados.index + 11

In [ ]:
# Crio um identificador único para rastrear esta execução
id_execucao = str(uuid4())

print("ID da execução:", id_execucao)


In [ ]:
dados[
    [
        "linha_origem",
        "CNPJ",
        "CEP",
    ]
].head()

In [ ]:
dados.head()

## Qualidade do CNPJ

Nesta etapa preservo o CNPJ recebido da fonte, aplico a normalização necessária para recuperar possíveis zeros à esquerda e valido os dígitos verificadores.

In [ ]:
# Preservo o valor original e acompanho cada etapa da transformação
dados["cnpj_original"] = dados["CNPJ"]

dados["cnpj_limpo"] = (
    dados["CNPJ"]
    .apply(limpar_cnpj)
)

dados["cnpj_normalizado"] = (
    dados["CNPJ"]
    .apply(normalizar_cnpj)
)

dados["cnpj_valido"] = (
    dados["cnpj_normalizado"]
    .apply(validar_cnpj)
)

In [ ]:
dados[
    [
        "cnpj_original",
        "cnpj_limpo",
        "cnpj_normalizado",
        "cnpj_valido",
    ]
].head(20)

In [ ]:
quantidade_cnpj_valido = dados["cnpj_valido"].sum()
quantidade_cnpj_invalido = len(dados) - quantidade_cnpj_valido

print(f"CNPJs válidos: {quantidade_cnpj_valido:,}")
print(f"CNPJs inválidos: {quantidade_cnpj_invalido:,}")
print(
    f"Percentual válido: "
    f"{quantidade_cnpj_valido / len(dados):.2%}"
)
print(
    f"Percentual inválido: "
    f"{quantidade_cnpj_invalido / len(dados):.2%}"
)

## Qualidade do CEP

Nesta etapa classifico os CEPs conforme a situação recebida da fonte.

CEPs com oito posições são mantidos, CEPs com sete posições podem ser normalizados com zero à esquerda e valores suspeitos não são corrigidos automaticamente.

In [ ]:
# Preservo o CEP recebido originalmente da fonte
dados["cep_original"] = dados["CEP"]

# Classifico cada registro antes da transformação
dados["status_cep"] = (
    dados["CEP"]
    .apply(classificar_cep)
)

# Normalizo somente os casos cuja correção é considerada segura
dados["cep_normalizado"] = (
    dados["CEP"]
    .apply(normalizar_cep)
)

# Registro se o CEP pode seguir após a validação estrutural
dados["cep_formato_valido"] = (
    dados["CEP"]
    .apply(validar_formato_cep)
)

In [ ]:
dados[
    [
        "cep_original",
        "status_cep",
        "cep_normalizado",
        "cep_formato_valido",
        "MUNICÍPIO",
        "ESTADO",
    ]
].head(20)

In [ ]:
dados["status_cep"].value_counts()

In [ ]:
dados["cep_formato_valido"].value_counts()

In [ ]:
quantidade_cep_valido = dados["cep_formato_valido"].sum()
quantidade_cep_rejeitado = len(dados) - quantidade_cep_valido

print(f"CEPs estruturalmente válidos: {quantidade_cep_valido:,}")
print(f"CEPs rejeitados: {quantidade_cep_rejeitado:,}")
print(
    f"Percentual válido: "
    f"{quantidade_cep_valido / len(dados):.4%}"
)
print(
    f"Percentual rejeitado: "
    f"{quantidade_cep_rejeitado / len(dados):.4%}"
)

## Aprovação e quarentena dos registros

Após validar individualmente CNPJ e CEP, classifico cada registro conforme
sua aptidão para seguir para a camada Silver.

Registros aprovados continuam no fluxo de transformação. Registros que
violam regras críticas são preservados em quarentena junto com o motivo
da rejeição, permitindo auditoria, investigação e possível reprocessamento.

In [ ]:
resultado_qualidade = dados.apply(
    lambda linha: avaliar_qualidade_registro(
        linha["cnpj_valido"],
        linha["cep_formato_valido"],
    ),
    axis=1,
    result_type="expand",
)

resultado_qualidade.columns = [
    "status_qualidade",
    "motivo_quarentena",
]

dados[
    [
        "status_qualidade",
        "motivo_quarentena",
    ]
] = resultado_qualidade

In [ ]:
dados["status_qualidade"].value_counts()

In [ ]:
dados["motivo_quarentena"].value_counts(dropna=False)

In [ ]:
# Separo os registros que podem seguir para a futura camada Silver
dados_aprovados = (
    dados[
        dados["status_qualidade"] == "APROVADO"
    ]
    .copy()
)

# Preservo separadamente os registros rejeitados para investigação
dados_quarentena = preparar_quarentena(
    dados=dados,
    id_execucao=id_execucao,
    arquivo_origem=caminho_arquivo.name,
)

In [ ]:
dados_quarentena[
    [
        "linha_origem",
        "CNPJ",
        "RAZÃO",
        "ENDEREÇO",
        "CEP",
        "cep_normalizado",
        "MUNICÍPIO",
        "ESTADO",
        "PRODUTO",
        "status_qualidade",
        "motivo_quarentena",
        "id_execucao",
        "arquivo_origem",
        "data_processamento_utc",
    ]
]


In [ ]:
print(f"Registros recebidos: {len(dados):,}")
print(f"Registros aprovados: {len(dados_aprovados):,}")
print(f"Registros em quarentena: {len(dados_quarentena):,}")

## Conclusão

A amostra analisada possui 19.516 registros.

Após a aplicação das regras de qualidade para CNPJ e CEP:

- 19.515 registros foram aprovados para continuidade no fluxo de dados;
- 1 registro foi direcionado para quarentena devido a um CEP suspeito;
- os registros rejeitados são preservados com informações de rastreabilidade, incluindo linha de origem, arquivo, identificador da execução e data de processamento.

As regras validadas neste notebook foram implementadas em módulos reutilizáveis em `src/qualidade`, permitindo sua aplicação posterior no fluxo Bronze → Silver.
